# 03 — APTnotes: An Unlabeled Pool of Real Threat Reports

**Objective:** download a **sample** of real APT reports (PDFs), extract clean plain text, and save them as an **unlabeled pool** that notebook 09 (active learning) will pull from.

## What this notebook is — and isn't

Unlike DNRTI (notebook 01) and TRAM (notebook 02), **APTnotes has no labels**. It's just a curated index of public vendor APT reports (FireEye, Kaspersky, Mandiant, …) published as PDFs.

We use it for one specific purpose: **active learning**. In notebook 09, we'll take our tactic classifier (trained on TRAM in notebook 07), run it on these unlabeled reports, measure how *uncertain* the model is on each one, and use that to decide which reports are worth hand-labeling next.

This notebook **does no training and no labeling.** It's data plumbing:

1. Download the APTnotes index
2. Pick a random sample of ~100 reports (full corpus is ~650 PDFs, several GB — too slow for iteration)
3. Fetch those PDFs
4. Extract plain text from each PDF
5. Save as a `Dataset` in `processed/aptnotes_pool/`

## Why sample instead of taking all 650?

- **Speed:** downloading and text-extracting 650 PDFs takes an hour+; 100 takes a few minutes.
- **Token cost during dev:** smaller pool = faster iteration on notebook 09 logic.
- **Realistic for the exercise:** 100 unlabeled reports is already more than a human analyst would label in a sitting, which is the whole point of active learning.

Scale up later by changing one constant (`N_SAMPLE`).

## Legal note

APTnotes PDFs are **vendor-copyrighted** (each belongs to its publishing company). Fine for local research use. **Do not redistribute** — the `.gitignore` keeps `data/aptnotes/` out of git.

## Step 1 — Fetch the APTnotes index

The current repo is `aptnotes/data`. It contains two index files: `APTnotes.csv` and `APTnotes.json`. Each entry has a filename, title, source, publication date, SHA-1, and a **Box.com download link**.

We grab just the JSON index directly — no need to clone the whole repo.

In [4]:
import json
import urllib.request
from pathlib import Path

DATA_DIR = Path('data/aptnotes')
PDF_DIR = DATA_DIR / 'pdfs'
TEXT_DIR = DATA_DIR / 'texts'
INDEX_PATH = DATA_DIR / 'APTnotes.json'
INDEX_URL = 'https://raw.githubusercontent.com/aptnotes/data/master/APTnotes.json'

PDF_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

if not INDEX_PATH.exists():
    urllib.request.urlretrieve(INDEX_URL, INDEX_PATH)
    print(f'Downloaded index to {INDEX_PATH}')
else:
    print(f'Index already cached at {INDEX_PATH}')

with open(INDEX_PATH) as f:
    index = json.load(f)

print(f'\nTotal reports in index: {len(index)}')
print(f'Example entry keys: {list(index[0].keys())}')
print(f'\nFirst entry:')
for k, v in index[0].items():
    val = str(v)
    print(f'  {k:14s} {val[:100]}')

Index already cached at data/aptnotes/APTnotes.json

Total reports in index: 689
Example entry keys: ['Filename', 'Title', 'Source', 'Link', 'SHA-1', 'Date', 'Year']

First entry:
  Filename       WickedRose_andNCPH
  Title          "Wicked Rose" And The Ncph Hacking Group
  Source         iDefense
  Link           https://app.box.com/s/0cp8nyd339dnbak96x2klgz1kxm36xd2
  SHA-1          e802d6f06085f22242a66b06c785315e1c63c070
  Date           12/01/2006
  Year           2006


### What's in each index record

Typical fields (names vary slightly between index versions):

| Field | Meaning |
|---|---|
| `Filename` | PDF filename |
| `Title` | Human-readable report title |
| `Source` | Publisher (FireEye, Kaspersky, …) |
| `Date` | Publication date |
| `Link` | Box.com download URL |
| `SHA-1` | Hash for integrity check |

We'll use `Link` for downloading and `Filename` as the local filename.

## Step 2 — Pick a random sample

100 reports is enough for active-learning experiments. We seed the random choice so this notebook is reproducible: rerunning always gives the same sample.

In [5]:
import random

N_SAMPLE = 100
random.seed(42)

# Probe for link/filename keys (APTnotes has used both 'Link' and 'link', etc.)
def pick_key(rec, candidates):
    for k in candidates:
        if k in rec:
            return k
    return None

LINK_KEY = pick_key(index[0], ['Link', 'link', 'URL', 'url'])
NAME_KEY = pick_key(index[0], ['Filename', 'filename', 'File Name', 'file_name'])
TITLE_KEY = pick_key(index[0], ['Title', 'title'])
SOURCE_KEY = pick_key(index[0], ['Source', 'source'])

print(f'Index field mapping: link={LINK_KEY!r}, filename={NAME_KEY!r}, '
      f'title={TITLE_KEY!r}, source={SOURCE_KEY!r}')

# Only keep records with both a link and a filename
usable = [r for r in index if r.get(LINK_KEY) and r.get(NAME_KEY)]
print(f'Usable records: {len(usable)} / {len(index)}')

sample = random.sample(usable, min(N_SAMPLE, len(usable)))
print(f'\nSampled {len(sample)} reports. First few titles:')
for r in sample[:5]:
    title = r.get(TITLE_KEY, '(no title)')
    source = r.get(SOURCE_KEY, '?')
    print(f'  [{source}] {title[:80]}')

Index field mapping: link='Link', filename='Filename', title='Title', source='Source'
Usable records: 689 / 689

Sampled 100 reports. First few titles:
  [Symantec] Hydrochasma: Previously Unknown Group Targets Medical and Shipping Organizations
  [Fidelis] New Cdto: A Sneakernet Trojan Solution
  [Zscaler, ThreatLabz] Alleged Apt Intrusion Set: 1.Php Group
  [Palo Alto] Watering Hole Attack On Aerospace Firm Exploits CVE-2015-5122 To Install Isspace
  [Kaspersky] The Cozyduke APT


## Step 3 — Download the sampled PDFs (via Box shared-link trick)

APTnotes' `Link` field points to Box.com shared URLs like `https://app.box.com/s/<token>`. These are **not** direct PDF downloads — they serve an HTML landing page that loads the file via JavaScript. A naive `urllib.urlopen` just gets HTML back (which is why an earlier version of this cell produced 100% failures).

The fix is a two-step technique, borrowed from the official `aptnotes/tools` downloader:

1. **GET the shared URL.** Its HTML embeds a JSON blob assigned to `Box.postStreamData`. That blob contains the file's `sharedName` (same token as the URL) and a numeric `itemID`.
2. **Build a direct-download URL** from those two values:
   ```
   https://app.box.com/index.php?rm=box_download_shared_file&shared_name=<sharedName>&file_id=f_<itemID>
   ```
3. **GET that URL.** Box now streams the raw PDF bytes back.

No Box login, no API key. We use the `requests` library for clean session + streaming behavior. If missing: `pip install requests`.

Expect ~5–10% failures (DMCA takedowns, Box rate limits, occasional layout changes). We log and skip failures. Safe to re-run — existing PDFs are skipped.

In [6]:
import re
import time
import requests

BOX_DATA_RE = re.compile(r'Box\.postStreamData\s*=\s*(\{.*?\});', re.DOTALL)
SHARED_ITEM_KEY = '/app-api/enduserapp/shared-item'


def resolve_box_download(session: requests.Session, share_url: str) -> str | None:
    """Turn a Box shared URL into a direct-download URL, or None on failure."""
    try:
        r = session.get(share_url, timeout=30)
        if r.status_code != 200:
            return None
        m = BOX_DATA_RE.search(r.text)
        if not m:
            return None
        blob = json.loads(m.group(1))
        item = blob.get(SHARED_ITEM_KEY)
        if not item or not item.get('sharedName') or not item.get('itemID'):
            return None
        return (
            'https://app.box.com/index.php?rm=box_download_shared_file'
            f'&shared_name={item["sharedName"]}'
            f'&file_id=f_{item["itemID"]}'
        )
    except (requests.RequestException, ValueError):
        return None


def fetch_pdf(session: requests.Session, share_url: str, out_path: Path,
              timeout: int = 60) -> bool:
    dl_url = resolve_box_download(session, share_url)
    if not dl_url:
        return False
    try:
        with session.get(dl_url, stream=True, timeout=timeout) as r:
            if r.status_code != 200:
                return False
            # Write to a temp path first so a partial download can't masquerade as complete.
            tmp = out_path.with_suffix(out_path.suffix + '.part')
            with open(tmp, 'wb') as f:
                for chunk in r.iter_content(1 << 14):
                    if chunk:
                        f.write(chunk)
            with open(tmp, 'rb') as f:
                head = f.read(8)
            if not head.startswith(b'%PDF'):
                tmp.unlink(missing_ok=True)
                return False
            tmp.replace(out_path)
            return True
    except (requests.RequestException, OSError):
        return False


session = requests.Session()
session.headers['User-Agent'] = 'Mozilla/5.0'

downloaded, skipped, failed = [], [], []

for i, rec in enumerate(sample, 1):
    filename = rec[NAME_KEY]
    if not filename.lower().endswith('.pdf'):
        filename += '.pdf'
    out_path = PDF_DIR / filename

    if out_path.exists() and out_path.stat().st_size > 0:
        skipped.append(filename)
        continue

    ok = fetch_pdf(session, rec[LINK_KEY], out_path)
    if ok:
        downloaded.append(filename)
    else:
        failed.append(filename)

    # gentle pacing to avoid Box rate-limits
    time.sleep(0.3)

    if i % 10 == 0:
        print(f'  progress: {i}/{len(sample)}  '
              f'downloaded={len(downloaded)} skipped={len(skipped)} failed={len(failed)}')

print(f'\nFinal: downloaded={len(downloaded)}  already-had={len(skipped)}  failed={len(failed)}')
if failed:
    print(f'First few failures: {failed[:5]}')

  progress: 10/100  downloaded=10 skipped=0 failed=0
  progress: 20/100  downloaded=20 skipped=0 failed=0
  progress: 30/100  downloaded=30 skipped=0 failed=0
  progress: 40/100  downloaded=40 skipped=0 failed=0
  progress: 50/100  downloaded=50 skipped=0 failed=0
  progress: 60/100  downloaded=60 skipped=0 failed=0
  progress: 70/100  downloaded=70 skipped=0 failed=0
  progress: 80/100  downloaded=80 skipped=0 failed=0
  progress: 90/100  downloaded=90 skipped=0 failed=0
  progress: 100/100  downloaded=100 skipped=0 failed=0

Final: downloaded=100  already-had=0  failed=0


**Interpreting the output:**

- A handful of failures (<10) is normal — dead Box links, DMCA'd reports.
- Many failures (>30) usually means Box changed their page structure. Open one of the failing links in a browser — if it loads for a human, regex `Box\.postStreamData` has drifted. Inspect the page source and update the regex / keys in `resolve_box_download`.
- Zero downloads means `requests` isn't installed, you're offline, or Box is blocking your IP entirely.

Re-running the cell is safe: successfully downloaded PDFs sit on disk and will be recognized by the `out_path.exists()` check.

## Step 4 — Extract text from PDFs

We use `pdfplumber`, which handles most threat-report layouts reasonably well (multi-column, tables). For each PDF, we concatenate the text from every page with double newlines.

PDFs are messy: some are scanned images (no extractable text), some have bizarre layouts. We catch errors and skip unreadable files rather than crashing.

In [7]:
import pdfplumber
import warnings

# pdfplumber emits lots of low-level warnings on malformed PDFs — silence them.
warnings.filterwarnings('ignore')

def pdf_to_text(pdf_path: Path) -> str | None:
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages = []
            for page in pdf.pages:
                txt = page.extract_text() or ''
                pages.append(txt)
        return '\n\n'.join(pages).strip()
    except Exception:
        return None

pdfs = sorted(PDF_DIR.glob('*.pdf'))
print(f'Found {len(pdfs)} PDFs on disk.')

extracted, empty, errored = 0, 0, 0
for i, pdf_path in enumerate(pdfs, 1):
    txt_path = TEXT_DIR / (pdf_path.stem + '.txt')
    if txt_path.exists():
        extracted += 1
        continue

    text = pdf_to_text(pdf_path)
    if text is None:
        errored += 1
        continue
    if len(text) < 200:
        # Scanned/image-only PDFs typically produce near-empty strings.
        empty += 1
        continue

    txt_path.write_text(text, encoding='utf-8')
    extracted += 1

    if i % 10 == 0:
        print(f'  progress: {i}/{len(pdfs)}  '
              f'extracted={extracted} empty={empty} errored={errored}')

print(f'\nFinal: extracted={extracted}  empty-or-scanned={empty}  errored={errored}')

Found 100 PDFs on disk.
  progress: 10/100  extracted=10 empty=0 errored=0
  progress: 20/100  extracted=20 empty=0 errored=0
  progress: 30/100  extracted=30 empty=0 errored=0
  progress: 40/100  extracted=40 empty=0 errored=0
  progress: 50/100  extracted=50 empty=0 errored=0
  progress: 60/100  extracted=60 empty=0 errored=0
  progress: 70/100  extracted=70 empty=0 errored=0
  progress: 80/100  extracted=80 empty=0 errored=0
  progress: 90/100  extracted=90 empty=0 errored=0
  progress: 100/100  extracted=100 empty=0 errored=0

Final: extracted=100  empty-or-scanned=0  errored=0


## Step 5 — Inspect what we got

Quick sanity checks:

- How long are these reports? (Context-length preview for notebook 08 / Longformer.)
- Does the text look like English threat-report prose, or garbled PDF artifacts?

In [8]:
import numpy as np

text_files = sorted(TEXT_DIR.glob('*.txt'))
print(f'Extracted text files: {len(text_files)}\n')

word_counts = []
for p in text_files:
    txt = p.read_text(encoding='utf-8')
    word_counts.append(len(txt.split()))

arr = np.array(word_counts)
print('Report length (in whitespace words):')
print(f'  min    {arr.min()}')
print(f'  median {int(np.median(arr))}')
print(f'  mean   {arr.mean():.0f}')
print(f'  p95    {int(np.percentile(arr, 95))}')
print(f'  max    {arr.max()}')

# Reminder: BERT max_length is 512 subword tokens (~380 words).
# Longformer goes to 4096. Notebook 08 compares chunking vs Longformer on these.
print(f'\n% of reports over 512 words:  {(arr > 512).mean()*100:.0f}%')
print(f'% of reports over 4096 words: {(arr > 4096).mean()*100:.0f}%')

Extracted text files: 100

Report length (in whitespace words):
  min    300
  median 2651
  mean   3969
  p95    10135
  max    37101

% of reports over 512 words:  98%
% of reports over 4096 words: 36%


In [9]:
# Peek at one report
sample_text = text_files[0].read_text(encoding='utf-8')
print(f'File: {text_files[0].name}')
print(f'Length: {len(sample_text)} chars, ~{len(sample_text.split())} words\n')
print('--- First 800 chars ---')
print(sample_text[:800])
print('\n--- Last 400 chars ---')
print(sample_text[-400:])

File: 2013-9.txt
Length: 119271 chars, ~18412 words

--- First 800 chars ---
CERIAS Tech Report 2013-9
Crude Faux: An analysis of cyber conflict within the oil & gas industries
by Kambic, K., Aurthor, K,. Ellis, W., Jensen, T., Johansen, K., Lee, B., Liles, S.
Center for Education and Research
Information Assurance and Security
Purdue University, West Lafayette, IN 47907-2086

Crude Faux
An Analysis of cyber Conflict Within the Oil & Gas Industries
Authors Abstract
Jake Kambic, Kristine Aurthor, Will The oil & gas industry is a mul-
Ellis, Mary Horner, Tyler Jensen, tibillion-dollar industry that has
Kyle Johansen, Brian Lee a history of conflict. As modern
technology has developed, both the
corporate aspects and technical as-
Under the direction of pects of the oil & gas industry have
Dr. Samuel Liles become heavily reliant on the Cy-
ber domain. The inherently in

--- Last 400 chars ---
e, R A and Knake, R K (2010). CyberWar, the next threat to national security and what to do about 

If the sample looks like real threat-report prose (executive summary, indicators, technique descriptions), we're good. If it looks like garbled characters, the PDF was likely scanned — drop it or OCR it separately (out of scope here).

## Step 6 — Save as an unlabeled `Dataset`

We store the extracted text plus lightweight metadata (filename, title, source) so notebook 09 can show the analyst *which* report the model is uncertain about — not just "sample #42".

In [10]:
from datasets import Dataset, Features, Value

# Build a lookup from filename stem -> index record, for metadata
by_stem = {}
for rec in index:
    fname = rec.get(NAME_KEY, '')
    stem = Path(fname).stem if fname else ''
    if stem:
        by_stem[stem] = rec

rows = {'filename': [], 'title': [], 'source': [], 'text': []}
for p in text_files:
    rec = by_stem.get(p.stem, {})
    rows['filename'].append(p.stem)
    rows['title'].append(rec.get(TITLE_KEY, '') or '')
    rows['source'].append(rec.get(SOURCE_KEY, '') or '')
    rows['text'].append(p.read_text(encoding='utf-8'))

features = Features({
    'filename': Value('string'),
    'title': Value('string'),
    'source': Value('string'),
    'text': Value('string'),
})

pool = Dataset.from_dict(rows, features=features)

OUT_DIR = Path('processed/aptnotes_pool')
pool.save_to_disk(str(OUT_DIR))

print(f'Saved {len(pool)} reports to {OUT_DIR}')
print(pool)

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saved 100 reports to processed/aptnotes_pool
Dataset({
    features: ['filename', 'title', 'source', 'text'],
    num_rows: 100
})


## Step 7 — Smoke test: load it back

In [11]:
from datasets import load_from_disk

reloaded = load_from_disk(str(OUT_DIR))
ex = reloaded[0]
print(f'filename: {ex["filename"]}')
print(f'title:    {ex["title"][:80]}')
print(f'source:   {ex["source"]}')
print(f'text (first 300 chars): {ex["text"][:300]}...')

filename: 2013-9
title:    Crude Faux: An Analysis Of Cyber Conflict Within The Oil & Gas Industries
source:   CERIAS
text (first 300 chars): CERIAS Tech Report 2013-9
Crude Faux: An analysis of cyber conflict within the oil & gas industries
by Kambic, K., Aurthor, K,. Ellis, W., Jensen, T., Johansen, K., Lee, B., Liles, S.
Center for Education and Research
Information Assurance and Security
Purdue University, West Lafayette, IN 47907-208...


## Summary

| | What |
|---|---|
| Source | APTnotes (`aptnotes/data`) — public vendor APT reports |
| Labels | **None** (this is an unlabeled pool) |
| Sample size | ~100 reports |
| Format | `Dataset` with `filename`, `title`, `source`, `text` |
| Saved to | `processed/aptnotes_pool/` |

## Where this fits in the bigger picture

- **Notebooks 01 + 04, 05** → **NER track**, using DNRTI (labeled).
- **Notebooks 02 + 06, 07** → **classification track**, using TRAM (labeled).
- **Notebook 03 (this one)** → feeds **notebook 08** (Longformer on long full-report inputs) and **notebook 09** (active learning — picking which of these unlabeled reports to label next).

**Next:** notebook 04 trains a NER model on DNRTI.